In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pygame
import random

class BreakoutEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}

    def __init__(self, render_mode=None):
        self.render_mode = render_mode
        self.window_width = 600
        self.window_height = 400
        
        self.paddle_width = 80
        self.paddle_height = 10
        self.ball_radius = 5
        self.num_bricks_x = 5
        self.num_bricks_y = 3
        self.brick_offset_y = 40

        self.wall_width = 50
        self.wall_height = 10

        self.total_bricks = self.num_bricks_x * self.num_bricks_y
        
        self.action_space = spaces.Discrete(3)
        
        obs_dim = 5 + self.total_bricks
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )
        
        self.window = None
        self.clock = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.paddle_x = self.window_width / 2 - self.paddle_width / 2
        self.ball_x = self.window_width / 2
        self.ball_y = self.window_height - 30
        
        self.ball_vx = self.np_random.choice([-4.0, 4.0])
        self.ball_vy = -4.0
        
        self.bricks = np.ones(self.total_bricks, dtype=np.float32)


        max_walls = self.window_width // self.wall_width
        self.num_walls = 5
        self.walls_offset = 150
        self.walls = [1] * self.num_walls + [0] * (max_walls - self.num_walls)
        random.shuffle(self.walls)

        info = {}
        return self._get_obs(), info

    def _get_obs(self):
        state = [self.paddle_x, self.ball_x, self.ball_y, self.ball_vx, self.ball_vy]
        state.extend(self.bricks)
        return np.array(state, dtype=np.float32)

    def step(self, action):
        reward = 0.0
        terminated = False
        truncated = False
        
        paddle_speed = 8.0
        if action == 1:
            self.paddle_x -= paddle_speed
        elif action == 2:
            self.paddle_x += paddle_speed

        # Ograniczenie ruchu paletki do krawędzi ekranu
            
        self.paddle_x = np.clip(self.paddle_x, 0, self.window_width - self.paddle_width)


        # Ruch piłki
        prev_ball_x = self.ball_x
        prev_ball_y = self.ball_y

        self.ball_x += self.ball_vx
        self.ball_y += self.ball_vy
        
        # Odbicie od krawędzi planszy

        if self.ball_x - self.ball_radius <= 0:
            self.ball_x = self.ball_radius
            self.ball_vx = abs(self.ball_vx)
        elif self.ball_x + self.ball_radius >= self.window_width:
            self.ball_x = self.window_width - self.ball_radius
            self.ball_vx = -abs(self.ball_vx)
            
        if self.ball_y - self.ball_radius <= 0:
            self.ball_y = self.ball_radius
            self.ball_vy = abs(self.ball_vy)

        # Odbicie od wewnętrznych ścian
        for i, wall in enumerate(self.walls):
            if wall != 1:
                continue

            wall_x = i * self.wall_width
            wall_y = self.walls_offset

            if (self.ball_x + self.ball_radius < wall_x or
                self.ball_x - self.ball_radius > wall_x + self.wall_width or
                self.ball_y + self.ball_radius < wall_y or
                self.ball_y - self.ball_radius > wall_y + self.wall_height):
                continue

            hit_from_top = (
                prev_ball_y + self.ball_radius <= wall_y and
                self.ball_y + self.ball_radius >= wall_y
            )
            hit_from_bottom = (
                prev_ball_y - self.ball_radius >= wall_y + self.wall_height and
                self.ball_y - self.ball_radius <= wall_y + self.wall_height
            )
            hit_from_left = (
                prev_ball_x + self.ball_radius <= wall_x and
                self.ball_x + self.ball_radius >= wall_x
            )
            hit_from_right = (
                prev_ball_x - self.ball_radius >= wall_x + self.wall_width and
                self.ball_x - self.ball_radius <= wall_x + self.wall_width
            )

            if hit_from_top:
                self.ball_y = wall_y - self.ball_radius
                self.ball_vy = -abs(self.ball_vy)
            elif hit_from_bottom:
                self.ball_y = wall_y + self.wall_height + self.ball_radius
                self.ball_vy = abs(self.ball_vy)
            elif hit_from_left:
                self.ball_x = wall_x - self.ball_radius
                self.ball_vx = -abs(self.ball_vx)
            elif hit_from_right:
                self.ball_x = wall_x + self.wall_width + self.ball_radius
                self.ball_vx = abs(self.ball_vx)
            else:
                self.ball_vy = -self.ball_vy

            break

        # Odbicie od paletki

        if (self.ball_y + self.ball_radius >= self.window_height - self.paddle_height - 10) and \
           (self.paddle_x <= self.ball_x <= self.paddle_x + self.paddle_width) and \
           self.ball_vy > 0:
            self.ball_vy = -self.ball_vy
            reward += 1.0  

        # Sprawdzanie kolizji z cegiełkami
        brick_width = self.window_width / self.num_bricks_x
        brick_height = 20
        brick_offset_y = 40

        if brick_offset_y <= self.ball_y <= brick_offset_y + self.num_bricks_y * brick_height:
            brick_col = int(self.ball_x // brick_width)
            brick_row = int((self.ball_y - brick_offset_y) // brick_height)

            if 0 <= brick_col < self.num_bricks_x and 0 <= brick_row < self.num_bricks_y:
                brick_idx = brick_row * self.num_bricks_x + brick_col

                if self.bricks[brick_idx] == 1:
                    self.bricks[brick_idx] = 0
                    self.ball_vy = -self.ball_vy
                    reward += 5.0
        
        if np.sum(self.bricks) == 0:
            reward += 100.0
            terminated = True
            
        if self.ball_y >= self.window_height:
            reward -= 10.0
            terminated = True

        if self.step_count >= self.max_episode_steps:
            truncated = True

        if self.render_mode == "human":
            self.render()

        return self._get_obs(), reward, terminated, truncated, {}

    def render(self):
        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_width, self.window_height))
            pygame.display.set_caption("AI Breakout")
            self.clock = pygame.time.Clock()

        if self.window is not None:
            self.window.fill((0, 0, 0))

            brick_width = self.window_width / self.num_bricks_x
            brick_height = 20
            brick_offset_y = 40

            for row in range(self.num_bricks_y):
                for col in range(self.num_bricks_x):
                    brick_idx = row * self.num_bricks_x + col
                    if self.bricks[brick_idx] == 1:
                        x = col * brick_width
                        y = brick_offset_y + row * brick_height
                        pygame.draw.rect(
                            self.window,
                            (200, 50, 50),
                            (x + 1, y + 1, brick_width - 2, brick_height - 2),
                        )

            for i, wall in enumerate(self.walls):
                if wall == 1:
                    pygame.draw.rect(self.window, (0, 255, 255),
                             (i * self.wall_width, self.walls_offset, self.wall_width, self.wall_height))
            
            pygame.draw.rect(self.window, (0, 255, 0),
                             (self.paddle_x, self.window_height - 20, self.paddle_width, self.paddle_height))
            
            pygame.draw.circle(self.window, (255, 255, 255),
                               (int(self.ball_x), int(self.ball_y)), self.ball_radius)
            
            pygame.event.pump() 
            pygame.display.flip()
            self.clock.tick(self.metadata["render_fps"])

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()

In [7]:
def play_game(is_manual=False, is_human=False):
    if is_human:
        env = BreakoutEnv(render_mode="human")
    else:
        env = BreakoutEnv(render_mode="rgb_array")
    obs, info = env.reset()
    
    env.render()
    
    terminated = False
    truncated = False
    

    if is_manual:
        print("Steruj strzałkami w LEWO i PRAWO. Zamknij okno, aby wyjść.")
        

    while not (terminated or truncated):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                terminated = True
        if is_manual:
            
            keys = pygame.key.get_pressed()
        
            action = 0 
            
            if keys[pygame.K_LEFT]:
                action = 1
            elif keys[pygame.K_RIGHT]:
                action = 2
        else:
            action = env.action_space.sample()

        obs, reward, terminated, truncated, info = env.step(action)
        
        if reward > 0:
            print(f"Nagroda: +{reward}")
        elif reward < 0:
            print(f"Kara: {reward}")

    env.close()
    print("Koniec gry! Twój epizod się zakończył.")

In [8]:
play_game(is_manual=True, is_human=True)

2026-05-13 22:23:13.678 Python[46935:6667241] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/yv/k8rsc3cd0s54dw42sstxmbfm0000gn/T/org.python.python.savedState


Steruj strzałkami w LEWO i PRAWO. Zamknij okno, aby wyjść.

Nagroda: +5.0

Nagroda: +5.0

Nagroda: +5.0

Nagroda: +5.0

Nagroda: +1.0

Nagroda: +5.0

Kara: -10.0

Koniec gry! Twój epizod się zakończył.

## Trening PPO (Stable-Baselines3)



In [38]:
from pathlib import Path

import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy


PROJECT_ROOT = Path.cwd()
MODELS_DIR = PROJECT_ROOT / "models"
BEST_MODEL_DIR = PROJECT_ROOT / "best_model"
LOG_DIR = PROJECT_ROOT / "logs"

for d in (MODELS_DIR, BEST_MODEL_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)


def make_env(render_mode=None, seed=None, max_episode_steps=1_000_000):
    env = BreakoutEnv(render_mode=render_mode, max_episode_steps=max_episode_steps)
    env = Monitor(env)
    env.reset(seed=seed)
    return env

In [21]:
SEED = 42
TOTAL_TIMESTEPS = 2_000_000
MAX_EPISODE_STEPS = 1_000_000

train_env = make_env(render_mode=None, seed=SEED, max_episode_steps=MAX_EPISODE_STEPS)
eval_env = make_env(render_mode=None, seed=SEED + 1, max_episode_steps=MAX_EPISODE_STEPS)

checkpoint_callback = CheckpointCallback(
    save_freq=200_000,
    save_path=str(MODELS_DIR),
    name_prefix="ppo_breakout_ckpt",
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(BEST_MODEL_DIR),
    log_path=str(LOG_DIR),
    eval_freq=10_000,
    deterministic=True,
    render=False,
)

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    verbose=0,
    seed=SEED,
    tensorboard_log=str(LOG_DIR / "tensorboard"),
)

In [22]:
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[checkpoint_callback, eval_callback],
    progress_bar=True,
)

final_model_path = MODELS_DIR / "ppo_breakout_final"
model.save(str(final_model_path))

print(f"Zapisano model: {final_model_path}.zip")

/Users/robertjacak/Documents/inteligencja-obliczeniowa/.venv-wlasnesrodowisko/lib/python3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


Eval num_timesteps=10000, episode_reward=4.40 +/- 14.04
Episode length: 244.00 +/- 137.73
New best mean reward!
Eval num_timesteps=20000, episode_reward=-1.80 +/- 7.00
Episode length: 184.00 +/- 76.12
Eval num_timesteps=30000, episode_reward=-4.80 +/- 3.19
Episode length: 165.60 +/- 47.67
Eval num_timesteps=40000, episode_reward=-1.60 +/- 5.82
Episode length: 203.40 +/- 51.13
Eval num_timesteps=50000, episode_reward=-9.00 +/- 2.00
Episode length: 117.60 +/- 13.20
Eval num_timesteps=60000, episode_reward=-3.00 +/- 8.72
Episode length: 152.20 +/- 51.03
Eval num_timesteps=70000, episode_reward=-6.80 +/- 4.12
Episode length: 152.40 +/- 55.71
Eval num_timesteps=80000, episode_reward=-0.80 +/- 9.09
Episode length: 185.00 +/- 61.11
Eval num_timesteps=90000, episode_reward=-9.00 +/- 2.00
Episode length: 117.60 +/- 13.20
Eval num_timesteps=100000, episode_reward=-4.00 +/- 5.83
Episode length: 149.20 +/- 36.23
Eval num_timesteps=110000, episode_reward=-6.80 +/- 4.35
Episode length: 152.20 +/- 67

 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,000,896/2,000,000  [ 0:31:28 < 0:00:00 , 2,365 it/s ]

Zapisano model: /Users/robertjacak/Documents/inteligencja-obliczeniowa/models/ppo_breakout_final.zip


In [36]:
best_model_path = BEST_MODEL_DIR / "best_model.zip"

model_path = MODELS_DIR / "ppo_breakout_ckpt_1200000_steps.zip"

if not best_model_path.exists():
    raise FileNotFoundError(f"Nie znaleziono modelu: {best_model_path}")

best_model = PPO.load(str(best_model_path))
best_model1 = PPO.load(str(model_path))


In [40]:
demo_env = BreakoutEnv(render_mode="human",max_episode_steps=1_000_000)
obs, info = demo_env.reset(seed=2026)
done = False
episode_reward = 0.0

while not done:
    action, _states = best_model1.predict(obs, deterministic=False)
    obs, reward, terminated, truncated, info = demo_env.step(action)
    episode_reward += reward
    done = terminated or truncated

print(f"Nagroda w epizodzie demo (best model): {episode_reward:.2f}")
demo_env.close()

Nagroda w epizodzie demo (best model): 186.00
